# Watching over a power transformer: what generated time series are for

A grid operator runs power transformers. Each one logs its **load** and its **oil temperature** (`OT`) every hour, and the operator relies on two models:

- a **forecaster** that predicts the oil temperature of the next 24 hours, to anticipate overheating;
- an **anomaly detector** that must raise an alert when a transformer misbehaves (cooling degradation, cooling cycling), and stay silent otherwise.

The recorded data is real but narrow: two transformers, two years, **no failure at all**. The transformations of `kcai-data-sampling-timeseries` create the situations nobody recorded, and this notebook uses them to answer two questions:

1. **Forecasting**: how much worse does the forecast get when the history is degraded (noisy probe, coarser logging)?
2. **Anomaly detection**: which faults does the detector catch, from what size, and does it stay quiet on harmless variability?

**How this notebook works.** Everything is computed here, step by step, with the package's own objects:

1. the **loader** cuts the series into one-week windows, the samples;
2. the **transformation classes** turn a week into a new week, and say what they did;
3. a list of transformations, with swept parameters, turns the record into **test sets**;
4. the two models are evaluated on them;
5. at the end, the same test sets are produced by the **pipeline** from the YAML configs of `examples/config/`, which is how the package is used outside a notebook, and we check that they are the same windows.

If time series are new to you, start with `air-passengers-demo.ipynb` (Air Passengers): the same steps on a textbook series.

**Data.** [ETT](https://github.com/zhouhaoyi/ETDataset) (*Electricity Transformer Temperature*, Zhou et al., Informer, AAAI 2021), the reference benchmark of forecasting: `ETTh1` and `ETTh2`, hourly, July 2016 to June 2018. ETT is licensed **CC BY-ND 4.0**: the data is downloaded by `scripts/fetch_ett_sample.py` (the first cell runs it if needed), and the outputs of this notebook (generated windows, figures) are adapted material, **not to be committed or shared** outside the team.

**Models.** Deliberately simple stand-ins for the operator's models, in `ett_models.py` next to this notebook. They need scikit-learn, which is not a workspace dependency:

```bash
uv run --with scikit-learn --with matplotlib --with ipykernel jupyter lab
```

The figures are those of the slide deck (*the ETT benchmark*), saved as PNG in `examples/outputs/ett-demo/figures/`. Seed 0 everywhere.

## Setup

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyBboxPatch, Polygon, Rectangle
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import yaml

%config InlineBackend.figure_format = 'png'
plt.rcParams["font.size"] = 15  # the only change from matplotlib's defaults: readable on a slide
pd.set_option("display.max_colwidth", None)

# Work from the repository root: every path in the configs is relative to it.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
os.chdir(REPO)
sys.path.insert(0, str(REPO / "examples" / "notebooks"))

# The package's objects: the loader that cuts windows, the transformations, the sweeps, the pipeline.
from kcai_data_sampling_core.models.sweep import SweepConfig
from kcai_data_sampling_job.cli import run
from kcai_data_sampling_job.dataloaders.api.timeseries import TimeSeriesLoaderConfig, TimeSeriesParquetDataLoader
from kcai_data_sampling_timeseries.api.transformations import (
    AddSinusoid,
    ColoredNoise,
    Drift,
    GarchNoise,
    ImpulseNoise,
    Jitter,
    LowPass,
    OUNoise,
)

import ett_models as em

DATA = Path("examples/data/ett_sample/series.parquet")
if not DATA.exists():
    subprocess.run([sys.executable, "scripts/fetch_ett_sample.py"], check=True)

# Figures are also saved as PNG for the slides (git-ignored, ETT-derived: not to be shared).
FIGURES = Path("examples/outputs/ett-demo/figures")
FIGURES.mkdir(parents=True, exist_ok=True)


def save(fig, name):
    # Each figure has the size of its slot in the deck: no tight bounding box, the aspect is kept.
    fig.savefig(FIGURES / f"{name}.png", dpi=200)

## 1. The data

The data file is a parquet table with one row per transformer. Each row holds a `(T, 7)` array: T hours, seven channels, six loads (`HUFL`, `HULL`, `MUFL`, `MULL`, `LUFL`, `LULL`: high, middle and low, useful and useless) and the oil temperature `OT`, channel 6.

In [ ]:
table = pq.read_table(DATA).to_pandas()
table.assign(values=table["values"].map(lambda v: f"({len(v)}, {len(v[0])}): first hour {np.round(v[0], 1).tolist()}"))

This notebook follows the oil temperature alone. Everything the models learn comes from the first 70 % of each series; every result below is measured on windows that start after it, so no evaluated window overlaps the training data.

In [ ]:
SERIES = em.load_ot(DATA)  # {series_id: (T,) oil temperature}
ENDS = em.train_end(SERIES)  # first evaluation hour, per series
START = pd.Timestamp(table.start[0])

fig, axes = plt.subplots(2, 1, figsize=(12.2, 6.2), sharex=True, sharey=True)
for ax, (sid, ot) in zip(axes, SERIES.items()):
    t = pd.date_range(START, periods=len(ot), freq="h")
    ax.plot(t, ot, lw=0.5)
    ax.axvspan(t[ENDS[sid]], t[-1], color="gray", alpha=0.15, lw=0)
    ax.text(t[ENDS[sid]], 0.95, "  evaluation", transform=ax.get_xaxis_transform(), va="top", fontsize=13)
    ax.set_title(f"{sid}: oil temperature (°C)", loc="left")
fig.tight_layout()
save(fig, "fig_data")
plt.show()

## 2. Samples are cut, not given

In image classification a sample is given: one image, collected as such, with an annotation outside the pixels. A time series is one long record; **the samples are windows cut out of it**, and the cut is a choice.

The loader makes that choice explicit. Its configuration is the same block as in the YAML configs (`dataloaders.loaders`): which file, which column and channels, and how to cut. Here, the oil temperature only (`channels: [6]`), windows of **one week (168 hours)**, **one starting every day** (stride 24).

In [ ]:
LOADER = {
    "name": "ett_ot",
    "type": "timeseries_parquet",
    "path": "examples/data/ett_sample/series.parquet",
    "id_column": "series_id",
    "values_column": "values",
    "sampling_rate": 0.000277778,  # 1 / 3600 Hz, informative: parameters are per step (per hour)
    "channels": [6],  # OT only
    "load_batch_size": 64,
    "windows": {"mode": "consecutive", "window": 168, "stride": 24},
}


def select(loader):
    """Cut the series into windows: the loader's selection, as a list of batches."""
    config = TimeSeriesLoaderConfig.model_validate(loader)
    (selection,) = TimeSeriesParquetDataLoader(config.name, config).get_selections()
    selection.bootstrap(None)
    return list(selection)


def keys_of(batches):
    """Where every window comes from: one row per window, with its id."""
    keys = pd.concat([b.columns.to_pandas().assign(id=b.ids) for b in batches], ignore_index=True)
    return keys[["series_id", "start_window", "end_window", "id"]]


def window_at(batches, series_id, start):
    """The one-window batch of ``series_id`` that starts at hour ``start``."""
    for batch in batches:
        cols = batch.columns.to_pandas()
        hit = np.flatnonzero((cols.series_id == series_id) & (cols.start_window == start))
        if len(hit):
            return batch.row(int(hit[0]))
    raise KeyError((series_id, start))


BATCHES = select(LOADER)
KEYS = keys_of(BATCHES)
print(f"{len(KEYS)} windows in {len(BATCHES)} batches; a batch holds a (windows, hours, channels) array: {BATCHES[0].data.shape}")
KEYS.groupby("series_id").size().to_frame("windows")

Two series of 17,420 hours give 1,438 one-week windows. They **overlap** (two neighbours share six days) and they are **ordered in time**, so the split between training and evaluation is made in time, never at random. Each window has an id, derived from where it was cut: every sample generated from it will point back to it.

We follow one window through the notebook: a spring week of `ETTh2` starting on 2 April 2018 (hour 15,360), in the evaluation part, with a clear daily cycle.

In [ ]:
SID, WEEK = "ETTh2", 15360
WINDOW = window_at(BATCHES, SID, WEEK)
original = WINDOW.data[0, :, 0]  # (168,) °C
assert WEEK >= ENDS[SID] and (original != 0).all()  # evaluation part, none of the runs of zeros ETT has in places
print(f"{SID}, week starting {START + pd.Timedelta(hours=WEEK):%Y-%m-%d}, id {WINDOW.ids[0]}")

**One transformation, one new sample.** A transformation class is built from its parameters, then `transform` takes a batch of windows and returns one `Output` per window: the new window `x`, and everything needed to say what was done to it (`algorithm`, `params`, `seed`, and `parent_id`, the window it comes from).

Here, `drift`: a ramp of +0.11 °C per hour over the week, a **cooling degradation** (the oil warms up for the same load).

In [ ]:
degradation = Drift({"slope": 0.11})
(out,) = degradation.transform(WINDOW)
print("new window:", out.x.shape)
{k: v for k, v in out.row().items() if k in ("id", "parent_id", "algorithm", "params", "seed")}

What happens to the label is a choice of the use case, not of the transformation:

- **forecasting**: the label is the next 24 hours of the same series, inside the signal. To keep it, we transform the history only and read the true future from the record;
- **anomaly detection**: the record has no label at all. A transformation can stay "normal" (harmless variability) or **create** the label "abnormal" (a fault, with its type, size and position, all in `params`).

In [ ]:
def image_panel(ax):
    """Image classification: a picture and its annotation (a drawing, for the comparison)."""
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 6)
    ax.axis("off")
    ax.add_patch(Rectangle((0.6, 0.6), 5.2, 4.6, facecolor="none", edgecolor="black", lw=1.5))
    ax.add_patch(Circle((4.4, 3.9), 0.55, color="C1"))
    ax.add_patch(Polygon([(0.6, 0.6), (2.4, 3.3), (3.5, 2.0), (4.4, 2.8), (5.8, 0.6)], closed=True, color="C0"))
    ax.add_patch(FancyBboxPatch((6.5, 2.4), 3.1, 1.0, boxstyle="round,pad=0.12", facecolor="white", edgecolor="black", lw=1.2))
    ax.text(8.05, 2.9, 'label: "mountain"', ha="center", va="center", fontsize=13)
    ax.plot([5.85, 6.4], [2.9, 2.9], color="black", lw=1.2)
    ax.set_title("One image, one annotation", loc="left")


OT = SERIES[SID]
T = pd.date_range(START, periods=len(OT), freq="h")
fig, axes = plt.subplots(1, 3, figsize=(18.1, 3.3))
image_panel(axes[0])
ax = axes[1]
a, b = WEEK - 96, WEEK + 168 + 96
ax.plot(T[a:b], OT[a:b], color="gray", lw=1, alpha=0.6)
ax.plot(T[WEEK : WEEK + 168], OT[WEEK : WEEK + 168], label="history: 1 week")
ax.plot(T[WEEK + 168 : WEEK + 192], OT[WEEK + 168 : WEEK + 192], lw=2.5, label="next 24 h = label")
ax.xaxis.set_major_locator(mdates.DayLocator(interval=4))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.set_ylim(OT[a:b].min() - 3, OT[a:b].max() + 16)
ax.legend(loc="upper left", ncol=2, fontsize=12)
ax.set_title("A window cut out of the series", loc="left")
ax = axes[2]
ax.plot(original, label="recorded")
ax.plot(out.x[:, 0], label="generated: label abnormal, drift +0.11 °C/h")
ax.set_ylim(0, 75)
ax.set_xticks(range(0, 169, 24))
ax.legend(loc="upper left", fontsize=12)
ax.set_title("The same week, with a generated fault", loc="left")
fig.tight_layout(w_pad=2.5)
save(fig, "fig_samples")
plt.show()

## 3. Transformations as situations

Each transformation stands for something that happens to a real transformer:

| Transformation | Situation | Role in the detection test |
|---|---|---|
| `jitter`, `sigma` 0.3 °C | noisy temperature probe | normal: must **not** raise an alert |
| `low_pass`, `cutoff` 0.1 cycle/h | readings averaged by the logging system | normal |
| `drift`, `slope` in °C per hour | cooling degradation: the oil warms up for the same load | fault: **must** raise an alert |
| `add_sinusoid`, 1 cycle per 6 h | cooling system cycling on and off | fault |

`jitter` draws random numbers, so it needs a **seed**: each window draws from its own generator, seeded by the seed, the window's id and the parameters, so the same week always gets the same noise, whatever the batch it is in. The other three are deterministic.

In [ ]:
SITUATIONS = {
    "Noisy probe": Jitter({"sigma": 0.3, "seed": 0}),
    "Coarser logging": LowPass({"cutoff": 0.1}),
    "Cooling degradation": Drift({"slope": 0.11}),
    "Cooling cycling": AddSinusoid({"frequency": 0.1666667, "amplitude": 10.0}),  # one cycle every 6 hours
}
SITUATION_OUTPUTS = {title: t.transform(WINDOW)[0] for title, t in SITUATIONS.items()}

fig, axes = plt.subplots(2, 2, figsize=(12.0, 6.5), sharex=True, sharey=True)
for ax, (title, output) in zip(axes.flat, SITUATION_OUTPUTS.items()):
    ax.plot(original, label="recorded")
    ax.plot(output.x[:, 0], label="generated")
    ax.set_title(title, loc="left")
    ax.set_xticks(range(0, 169, 24))
for ax in axes[1]:
    ax.set_xlabel("hour of the week")
for ax in axes[:, 0]:
    ax.set_ylabel("°C")
axes[0, 0].legend(loc="upper right", ncol=2)
fig.tight_layout()
save(fig, "fig_week")
plt.show()

## 4. Time-series noises

`jitter` is white noise: every hour independent of the others. Real disturbances have a memory. Four noises take time into account, and each one is either **additive** (`x + n`: the same size whatever the level) or **multiplicative** (`x · (1 + n)`: proportional to the level), chosen with `mode`.

| Transformation | Situation | Parameters |
|---|---|---|
| `colored_noise` | drifting probe: pink (1/f) noise, slow wanderings dominate | `sigma` 1 °C, `exponent` 1, additive |
| `ou_noise` | fluctuating offset that comes back (Ornstein-Uhlenbeck), correlated over about 10 hours | `sigma` 1.5 °C, `theta` 0.1, additive |
| `garch_noise` | calm and agitated periods (GARCH volatility), proportional to the temperature | `sigma` 5 %, `alpha` 0.15, `beta` 0.8, multiplicative |
| `impulse_noise` | acquisition glitches: isolated spikes | `rate` 2 % of the hours, `amplitude` ±8 °C, additive |

In [ ]:
NOISES = {
    "Pink noise (1/f), additive": ColoredNoise({"sigma": 1.0, "exponent": 1.0, "seed": 0}),
    "Ornstein-Uhlenbeck, additive": OUNoise({"sigma": 1.5, "theta": 0.1, "seed": 0}),
    "GARCH, multiplicative": GarchNoise({"sigma": 0.05, "alpha": 0.15, "beta": 0.8, "mode": "multiplicative", "seed": 0}),
    "Impulse spikes, additive": ImpulseNoise({"rate": 0.02, "amplitude": 8.0, "seed": 0}),
}
NOISE_OUTPUTS = {title: t.transform(WINDOW)[0] for title, t in NOISES.items()}

fig, axes = plt.subplots(2, 2, figsize=(12.0, 6.5), sharex=True, sharey=True)
for ax, (title, output) in zip(axes.flat, NOISE_OUTPUTS.items()):
    ax.plot(original, label="recorded")
    ax.plot(output.x[:, 0], label="generated")
    ax.set_title(title, loc="left")
    ax.set_xticks(range(0, 169, 24))
for ax in axes[1]:
    ax.set_xlabel("hour of the week")
for ax in axes[:, 0]:
    ax.set_ylabel("°C")
axes[0, 0].legend(loc="upper right", ncol=2)
fig.tight_layout()
save(fig, "fig_noises")
plt.show()

## 5. Generated data is labelled data

Every output carries its own record: where it comes from and what was done to it. Written by the pipeline, these records are the rows of the **ledger**. Here are the eight weeks generated above from the spring week. ETT has no labelled failure: we did not need one, the ledger is the label (which fault, of what size, in which week).

In [ ]:
pd.DataFrame([
    {"source window": f"{SID}, hours {WEEK}–{WEEK + 167}", **{k: o.row()[k] for k in ("algorithm", "params", "seed", "parent_id", "id")}}
    for o in [*SITUATION_OUTPUTS.values(), *NOISE_OUTPUTS.values()]
])

## 6. From one week to test sets: sweeping parameters

One week and one parameter value tell a story; a test needs every week, and **a range of sizes** for each degradation and each fault. A sweep is written as a `SweepConfig`, the same block as in the YAML configs: an interval and a number of values. It expands into one transformation per value.

Then `generate` applies every transformation to every window and keeps one record per output: a small ledger, in memory.

In [ ]:
def sweep(make, **sweep_config):
    """One transformation per value of a sweep; ``make`` builds it from the value."""
    return [make(v) for v in SweepConfig(**sweep_config).values()]


#: The parameter each curve is drawn against.
SWEPT = {"jitter": "sigma", "low_pass": "cutoff", "drift": "slope", "add_sinusoid": "amplitude",
         "colored_noise": "sigma", "ou_noise": "sigma", "garch_noise": "sigma", "impulse_noise": "amplitude"}


def generate(batches, transformations):
    """Apply every transformation to every window: the records (a ledger) and the new windows."""
    rows, windows = [], []
    for batch in batches:
        keys = batch.columns.to_pandas()[["series_id", "start_window", "end_window"]].to_dict("records")
        for t in transformations:
            for key, o in zip(keys, t.transform(batch), strict=True):
                rows.append({**key, "algorithm": o.algorithm, "params": o.params, "seed": o.seed, "parent_id": o.parent_id, "id": o.id})
                windows.append(o.x[:, 0])
    ledger = pd.DataFrame(rows)
    ledger["value"] = [p[SWEPT[a]] for a, p in zip(ledger.algorithm, ledger.params)]  # the x axis of the curves
    return ledger, np.stack(windows)


def evaluation(ledger, windows):
    """Keep the generated windows whose source window starts after the training part."""
    keep = (ledger.start_window >= ledger.series_id.map(ENDS)).to_numpy()
    return ledger[keep].reset_index(drop=True), windows[keep]


# Anomaly detection: two normal variabilities, two faults swept over their size.
ANOMALY_TRANSFORMATIONS = [
    Jitter({"sigma": 0.3, "seed": 0}),
    LowPass({"cutoff": 0.1}),
    *sweep(lambda v: Drift({"slope": v}), range=[0.01, 0.16], samples=4),
    *sweep(lambda v: AddSinusoid({"frequency": 0.1666667, "amplitude": v}), range=[1.0, 16.0], samples=6),
]
ANOMALY_ALL, ANOMALY_ALL_WINDOWS = generate(BATCHES, ANOMALY_TRANSFORMATIONS)
ANOMALY, ANOMALY_WINDOWS = evaluation(ANOMALY_ALL, ANOMALY_ALL_WINDOWS)
print(f"{len(ANOMALY_TRANSFORMATIONS)} transformations x {len(KEYS)} windows = {len(ANOMALY_ALL)} generated weeks, "
      f"{len(ANOMALY)} in the evaluation part")
ANOMALY.groupby("algorithm", sort=False).value.unique().to_frame("swept values")

Forecasting degrades the history in two ways, swept: probe noise from 0 to 2 °C, and coarser logging (`low_pass`: the lower the cutoff, the smoother the readings). Same one-week windows: each is the history of a 24-hour forecast.

In [ ]:
FORECAST_TRANSFORMATIONS = [
    *sweep(lambda v: Jitter({"sigma": v, "seed": 0}), range=[0.0, 2.0], samples=5),
    *sweep(lambda v: LowPass({"cutoff": v}), range=[0.05, 0.25], samples=3),
]
FORECAST_ALL, FORECAST_ALL_WINDOWS = generate(BATCHES, FORECAST_TRANSFORMATIONS)
FORECAST, FORECAST_WINDOWS = evaluation(FORECAST_ALL, FORECAST_ALL_WINDOWS)
print(f"{len(FORECAST_TRANSFORMATIONS)} transformations x {len(KEYS)} windows = {len(FORECAST_ALL)} generated histories, "
      f"{len(FORECAST)} in the evaluation part")

## 7. Forecasting: how much does a degraded history cost?

Two forecasters predict the next 24 hours from a one-week history:

- **seasonal naive**: tomorrow is today;
- **lag ridge**: a ridge regression on the week, fitted on clean training windows.

Each generated window is a degraded history; the **true future is read from the original series** after the window's end, so it is never transformed. We measure the mean absolute error of the 24-hour forecast as the degradation grows.

In [ ]:
HISTORIES, FUTURES = em.training_set(SERIES, 168)  # clean (week, next 24 h) pairs from the training part
MODELS = {
    "lag ridge": em.LagRidge().fit(HISTORIES, FUTURES),
    "seasonal naive": em.SeasonalNaive().fit(HISTORIES, FUTURES),
}
ERRORS = em.forecast_errors(FORECAST, FORECAST_WINDOWS, SERIES, MODELS)
TABLE = ERRORS.groupby(["algorithm", "value", "model"]).mae.mean().unstack()
TABLE.round(2)

In [ ]:
PANELS = [
    ("jitter", "Noisy probe", "noise sigma (°C)", False),
    ("low_pass", "Coarser logging", "cutoff (cycles per hour), coarser →", True),
]

fig, axes = plt.subplots(1, 2, figsize=(12.4, 5.2), sharey=True)
for ax, (algorithm, title, xlabel, invert) in zip(axes, PANELS):
    curve = TABLE.loc[algorithm]
    for model in MODELS:
        ax.plot(curve.index, curve[model], marker="o", label=model)
    ax.set_title(title, loc="left")
    ax.set_xlabel(xlabel)
    if invert:
        ax.invert_xaxis()
axes[0].set_ylabel("forecast error, next 24 h (°C)")
axes[0].set_ylim(0, 3.4)
axes[0].legend(loc="lower right")
fig.tight_layout()
save(fig, "fig_forecasting")
plt.show()
clean, noisy = TABLE.loc[("jitter", 0.0), "lag ridge"], TABLE.loc[("jitter", 2.0), "lag ridge"]
print(f"lag ridge: {clean:.2f} °C on clean histories, {noisy:.2f} °C with a 2 °C probe noise ({100 * (noisy / clean - 1):+.0f} %)")

The error grows steadily with the probe noise, for both models. The ridge, the better model on clean data, is also the one that loses most when the logging gets coarser: it relies on the hour-to-hour detail that a smoothed history no longer has.

## 8. Anomaly detection: what does the detector see?

The detector predicts the last 24 hours of each week from the 144 hours before them; the anomaly score is how badly it does. It raises an alert above the 95th percentile of the scores on clean training weeks, so about 5 % of normal weeks raise an alert by construction.

ETT has **no labelled anomaly**. The label comes from the ledger instead: we know which fault was injected, of what size, into which window. The false-alarm rate is measured on the recorded evaluation weeks, straight from the selection.

In [ ]:
DETECTOR = em.SurpriseDetector(window=168).fit(SERIES)
ANOMALY["alert"] = DETECTOR.alerts(ANOMALY_WINDOWS)

RECORDED = np.concatenate([b.data[:, :, 0] for b in BATCHES])  # the 1,438 recorded weeks, in KEYS order
in_evaluation = (KEYS.start_window >= KEYS.series_id.map(ENDS)).to_numpy()
FALSE_ALARMS = DETECTOR.alerts(RECORDED[in_evaluation]).mean() * 100
RATES = ANOMALY.groupby(["algorithm", "value"]).alert.mean() * 100

print(f"alert threshold: {DETECTOR.threshold:.2f} °C mean error")
pd.DataFrame({
    "weeks": ["recorded weeks", "noisy probe (jitter 0.3)", "coarser logging (low_pass 0.1)"],
    "alert rate (%)": [FALSE_ALARMS, RATES.loc[("jitter", 0.3)], RATES.loc[("low_pass", 0.1)]],
}).round(1)

Harmless variability does not raise more alerts than the recorded weeks: the detector is quiet where it should be. Now the faults, from small to large:

In [ ]:
NATURAL_SLOPE = 0.07  # 95th percentile of weekly OT slopes on ETTh2, °C per hour


def fault_figure(rates, xlabel, name, natural=None):
    fig, ax = plt.subplots(figsize=(12.4, 5.9))
    if natural:
        ax.axvspan(0, natural, color="gray", alpha=0.15, lw=0)
        ax.text(natural / 2, 97, "natural\nweekly slopes", ha="center", va="top", fontsize=13)
    ax.plot(rates.index, rates.values, marker="o")
    ax.axhline(FALSE_ALARMS, color="gray", lw=1.2, ls="--")
    ax.text(rates.index.max(), FALSE_ALARMS + 2.5, f"false alarms on recorded weeks: {FALSE_ALARMS:.1f} %",
            ha="right", va="bottom", color="gray", fontsize=13)
    for x, y in rates.items():
        ax.annotate(f"{y:.0f} %", (x, y), textcoords="offset points", xytext=(0, 10), ha="center", fontsize=13)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("windows raising an alert (%)")
    ax.set_ylim(0, 112)
    ax.set_xticks(rates.index)
    fig.tight_layout()
    save(fig, name)
    plt.show()


fault_figure(RATES.loc["drift"], "drift (°C per hour)", "fig_drift", natural=NATURAL_SLOPE)

**Cooling degradation** is caught once it clearly exceeds what the weather and the load already do to the oil temperature: a drift inside the natural range is invisible, a strong one is caught most of the time.

In [ ]:
fault_figure(RATES.loc["add_sinusoid"], "cycling amplitude (°C), one cycle every 6 hours", "fig_cycling")

**The blind spot.** Cooling cycling is nearly invisible until it is very large. The detector sees the oscillation in the history and *extrapolates* it, instead of being surprised by it: a regular fault is a predictable one. This is a blind spot of every forecast-based detector, and the generated faults reveal it **before a real failure does**.

## 9. The same test sets, from a config

Outside a notebook, nobody writes these loops: a YAML config names the loader and the transformations, and the pipeline (`kcai-data-sampling process -p <config>`, or `run(config)` from Python) writes the ledger as a parquet table and one `.npy` file per generated window. Here is the anomaly config; its `dataloaders` and `transformations` blocks are the ones built by hand above.

In [ ]:
print(Path("examples/config/ett-anomaly.yaml").read_text())

We run the three configs and check that the pipeline wrote **the same windows** as this notebook: same ids, same values. The ids match because they are derived from the same things (the source window, the algorithm, its parameters and seed), and the values match because each window's noise is seeded by its own id.

In [ ]:
NOISES_ALL, NOISES_ALL_WINDOWS = generate(BATCHES, list(NOISES.values()))
IN_NOTEBOOK = {
    "ett-anomaly": (ANOMALY_ALL, ANOMALY_ALL_WINDOWS),
    "ett-forecasting": (FORECAST_ALL, FORECAST_ALL_WINDOWS),
    "ett-noises": (NOISES_ALL, NOISES_ALL_WINDOWS),
}
for name, (ledger, windows) in IN_NOTEBOOK.items():
    with open(f"examples/config/{name}.yaml") as f:
        config = yaml.safe_load(f)
    config["compute"] = {**config.get("compute", {}), "progress_bar": False}  # quiet in a notebook
    shutil.rmtree(f"examples/outputs/{name}", ignore_errors=True)
    run(config)
    root = Path(f"examples/outputs/{name}/ett_ot")
    written = pq.read_table(root / "ledger.parquet").to_pandas()
    mine = dict(zip(ledger.id, windows))
    same = set(written.id) == set(mine) and all(
        np.allclose(np.load(root / "payloads" / a)[:, 0], mine[i]) for i, a in zip(written.id, written.artifact)
    )
    print(f"{name}: {len(written)} rows in the ledger, the same windows as in the notebook: {same}")

written.head(3)  # the ledger of the noises run: one row per generated window

## 10. What this shows

- **A sample is a window**, cut by the loader; a transformation turns it into a new window and keeps a record of what it did.
- **The same transformations serve both use cases.** For forecasting they degrade the input to measure robustness; for detection they play either harmless variability or a fault. The scenario gives the meaning, not the transformation.
- **Every generated window is traceable**: the ledger records its source window, the algorithm, its parameters and seed. That is what turns generated data into labelled test data.
- **Sweeping a parameter gives a curve, not a score**: how fast the forecast degrades, from what size a fault is caught, and where the detector is blind.